# 16 — GoalScope qualitative demo: refusal-steered sentence

Clone of `13_goalscope_qualitative.ipynb` (same renderer, same paper-style callout interaction),
swapping the source from a code-generation prompt to a **refusal-direction-steered** sentence
(`notebooks/clock/21_steering_vector_verbalization.ipynb`'s `refusal` vector): a harmless request
("How do I bake a chocolate cake?") with the refusal direction added to its residual stream, so
the model's own continuation drifts toward declining/warning language. Does the 122-head
GoalScope read-out verbalize *that* drift, token by token?

**Interaction:** click a highlighted (scoped) token to pin its read-out — a numbered callout
appears, the token is boxed + dotted, and a curve is drawn. Click again (or the ✕) to unpin.
Multiple tokens can be pinned. Hover shows a transient preview.

Renderer: `src/goals/goalscope_viz.py`. Input = the `list[dict]` returned by
`apply_goalscope(..., include_noncode=True, show=False)`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
# resolve to the repo root so relative paths (head_sets/, notebooks/diff/visualizations/) work
# regardless of the notebook's launch cwd
while not os.path.isdir('src') and os.getcwd() != '/':
    os.chdir('..')
sys.path.append('.')
print('cwd:', os.getcwd())
from src.goals.goalscope_viz import goalscope_qualitative_html, show_goalscope_qualitative

## GoalScope on Qwen2.5-Coder-32B
Load the model + the 122 goal heads, exactly as in `13`. *(Heavy — needs a free GPU. Set `CUDA`.)*

In [ ]:
import os, torch
MODEL = 'Qwen/Qwen2.5-Coder-32B-Instruct'
CUDA  = '7'   # set to free GPU(s)
os.environ['CUDA_VISIBLE_DEVICES'] = CUDA
from src.models import ModelandTokenizer
from src.goals.goalscope import apply_goalscope
from scripts.evaluation.eval_goalscope import load_json_heads

slug = MODEL.split('/')[-1]
HEADS_JSON = f'head_sets/{slug}/goal__lam20_thr0.9_band25-56.json'  # 122 goal heads
mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16, device_map='auto',
                       attn_implementation='eager')
heads = load_json_heads(HEADS_JSON, mt.n_layer, mt.model.config.num_attention_heads)
print(f'{len(heads)} goal heads loaded')

## Refusal direction + steered generation

Same construction as `21_steering_vector_verbalization.ipynb`: diff-of-means (harmful −
harmless instruction activations) at `INJECT_LAYER`, computed fresh for this model from a small
hand-written topic-matched instruction set (request text only, no harmful content). Injected into
a **harmless** prompt — the classic demonstration that the vector alone induces refusal-like
language on an innocuous request.

In [ ]:
from src.functional import PatchSpec
from scripts.steering.steering_pilot import (
    INJECT_LAYER, REFUSAL_PAIRS, compute_diff_of_means_vector,
    make_residual_steering_patch_interface, clean_generate_with_steering,
)
from src.tokens import prepare_input
from rich import print as rich_print

harmful = [p[0] for p in REFUSAL_PAIRS]
harmless = [p[1] for p in REFUSAL_PAIRS]
refusal_vec = compute_diff_of_means_vector(mt, harmful, harmless, INJECT_LAYER)
print('refusal_vec norm:', refusal_vec.float().norm().item())

SOURCE_PROMPT = 'How do I bake a chocolate cake?'
ALPHA = 8.0   # from the stage-1 sanity sweep in steering_pilot.py -- the largest value that
             # still produces LEGIBLE refusal-flavored language before collapsing into a loop
GEN_TOKENS = 64

clean = clean_generate_with_steering(mt, SOURCE_PROMPT, INJECT_LAYER, refusal_vec, 0.0, max_new_tokens=GEN_TOKENS)
gen = clean_generate_with_steering(mt, SOURCE_PROMPT, INJECT_LAYER, refusal_vec, ALPHA, max_new_tokens=GEN_TOKENS)
rich_print(f'[bold]source:[/bold] {SOURCE_PROMPT!r}')
rich_print(f'[white]clean:[/white]   {clean!r}')
rich_print(f'[sea_green3]steered:[/sea_green3] {gen!r}')

## GoalScope read-out over the steered continuation

Score the pivot (source prompt's last token) + every steered-generated token, with the refusal
vector re-applied at each of those positions individually during the extraction forward pass (see
`21_steering_vector_verbalization.ipynb` for why a single `location=(layer, -1)` patch isn't
enough once more than one position is scored). `include_noncode=True` keeps the whole
prompt+generation in the render, with the un-scored prefix dimmed.

In [ ]:
layer_name = mt.layer_name_format.format(INJECT_LAYER)
scaled_vec = ALPHA * refusal_vec

p0 = prepare_input(prompts=SOURCE_PROMPT, tokenizer=mt)['input_ids'].shape[1]
enc = prepare_input(prompts=SOURCE_PROMPT + gen, tokenizer=mt)
ptot = enc['input_ids'].shape[1]
positions = [p0 - 1] + list(range(p0, ptot))  # pivot + all steered-generated tokens

patches = [PatchSpec(location=(layer_name, idx), patch=scaled_vec, strategy='add', on_pass='all')
          for idx in positions]
patch_interface = make_residual_steering_patch_interface(patches)

# Destination read-out probe -- EDIT HERE to try a different one.
# Plain, non-code destination context (from notebook 21) -- also confirmed to produce clearly
# refusal-flavored verbalizations ("Generating harmful and illegal content", ...), so we're
# dropping the code-flavored GOALSCOPE_SYS/GOALSCOPE_QUESTION default in favor of this one.
def _chat(system, user, assistant=""):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}, {"role": "assistant", "content": assistant}]
    kw = dict(add_generation_prompt=False, continue_final_message=True, tokenize=False)
    return mt.tokenizer.apply_chat_template(msgs, chat_template_kwargs={"enable_thinking": False}, **kw)
GOALSCOPE_PROMPT = _chat(
    "You are a helpful assistant.",
    "You are in the middle of generating a response. In one short phrase, what behavior or topic "
    "are you contemplating right now?",
)

# Swap back to the code-flavored default instead (uncomment to compare):
# GOALSCOPE_PROMPT = None

res = apply_goalscope(
    prompt=enc, mt=mt, selected_heads=heads, strategy='additive_norm',
    steer_layer_range=(35, 46), alpha=0.1, n_steer_tokens='all',
    max_new_tokens=64, token_positions=positions, include_noncode=True, show=False,
    goalscope_prompt=GOALSCOPE_PROMPT,
    patch_interface=patch_interface,
)
print(sum(1 for r in res if r.get('scoped')), 'scoped tokens')

In [ ]:
# Inline (interactive) + save a standalone .html for the paper.
import pathlib; pathlib.Path('notebooks/diff/visualizations').mkdir(parents=True, exist_ok=True)
goalscope_qualitative_html(res, title='refusal-steered — GoalScope', highlight=False,
                           save_path='notebooks/diff/visualizations/refusal_steering.html')
show_goalscope_qualitative(res, title='refusal-steered — GoalScope', highlight=False, height=1024)

## Alternate renderer: callouts float near their own token (not a fixed column)

`goalscope_qualitative_flow_html` — for natural text (not code), a callout defaults to sitting
directly above/below its own anchor token (whichever has room) instead of stacking in one fixed
left column, since the relevant token can be anywhere in a paragraph. Each pinned callout gets
▲▼◀▶ buttons to override its side; click the active direction again to go back to auto.

In [ ]:
from src.goals.goalscope_viz import goalscope_qualitative_flow_html, show_goalscope_qualitative_flow

goalscope_qualitative_flow_html(res, title='refusal-steered — GoalScope (flow layout)',
                                save_path='notebooks/diff/visualizations/refusal_steering_flow.html')
show_goalscope_qualitative_flow(res, title='refusal-steered — GoalScope (flow layout)', height=900)